# Benchmark Model Tabular Ordinal

Alur notebook:

1. baseline tanpa tuning (1 seed),
2. tuning Optuna dengan objective minimize validation MAE,
3. retrain best hyperparameter (5 seed),
4. laporan mean ± std dan visualisasi enam metrik.

Model: TabM, RealMLP-TD-S, XGBoost, LightGBM, dan CatBoost.

## 1. Setup Kaggle

In [ ]:
%pip install -q tabm optuna xgboost catboost lightgbm

import copy
import json
import random
import sys
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping as lgb_early_stopping
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

## 2. Konfigurasi

Ubah hanya cell ini untuk mengatur biaya eksperimen. N_TRIALS=20 cocok untuk percobaan; 30–50 trial lebih layak untuk hasil final.

In [ ]:
BASELINE_SEED = 42
TUNING_SEED = 42
FINAL_SEEDS = [42, 123, 2024, 3407, 9999]

N_TRIALS = 20
RUN_BASELINE = True
RUN_TUNING = True
RUN_FINAL = True

TABM_BATCH_SIZE = 256
TABM_MAX_EPOCHS = 150
TABM_PATIENCE = 15
GPU_ID = 0

DATASET_FILES = {
    "melbourne_airbnb": "oc10_melbourneAirbnb.csv",
    "student_performance": "oc08_studentPerformance.csv",
    "nhanes": "oc05_nhanes.csv",
    "esl": "oc06_esl.csv",
    "lev": "oc05_lev.csv",
}
TUNABLE_MODELS = ["TabM", "XGBoost", "LightGBM", "CatBoost"]
ALL_MODELS = ["TabM", "RealMLP-TD-S", "XGBoost", "LightGBM", "CatBoost"]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR = WORK_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT == 0:
    raise RuntimeError("GPU tidak ditemukan. Aktifkan accelerator GPU di Kaggle.")

DEVICE = torch.device(f"cuda:{GPU_ID}")
CATBOOST_DEVICES = ":".join(map(str, range(min(2, GPU_COUNT))))

print(f"DEVICE | PyTorch={DEVICE} | CatBoost={CATBOOST_DEVICES}")
for gpu_id in range(GPU_COUNT):
    print(f"GPU {gpu_id} | {torch.cuda.get_device_name(gpu_id)}")

## 3. Load dataset dan RealMLP

Dataset dicari otomatis di folder lokal, /kaggle/input, dan /kaggle/working. Source RealMLP diunduh dari repository resminya karena tidak tersedia sebagai package pip.

In [ ]:
def find_dataset(filename):
    roots = [Path("dataset"), Path("/kaggle/input"), Path("/kaggle/working")]
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"{filename} tidak ditemukan")


datasets = {
    name: pd.read_csv(find_dataset(filename))
    for name, filename in DATASET_FILES.items()
}

for name, df in datasets.items():
    if "y" not in df.columns:
        raise ValueError(f"{name}: kolom target 'y' tidak ditemukan")
    classes = sorted(df["y"].unique())
    if classes != list(range(len(classes))):
        raise ValueError(f"{name}: label harus berurutan dari 0, ditemukan {classes}")
    print(f"{name:22} shape={df.shape}, classes={len(classes)}")


def load_realmlp():
    folder = WORK_DIR / "_realmlp"
    folder.mkdir(exist_ok=True)
    for filename in ["preprocessing.py", "mlp.py"]:
        path = folder / filename
        if not path.exists():
            url = (
                "https://raw.githubusercontent.com/"
                "dholzmueller/realmlp-td-s_standalone/main/"
                f"{filename}"
            )
            urllib.request.urlretrieve(url, path)

    np.NaN = np.nan
    np.Inf = np.inf
    sys.path.insert(0, str(folder.resolve()))
    from mlp import Standalone_RealMLP_TD_S_Classifier
    return Standalone_RealMLP_TD_S_Classifier


RealMLPClassifier = load_realmlp()

## 4. Split dan metrik ordinal

Split train/validation/test adalah 70/15/15 dan stratified. Test set tidak dipakai untuk tuning.

R², Spearman, MAE, dan RMSE memakai expected class dari probabilitas. QWK dan Accuracy memakai expected class yang dibulatkan dan di-clip ke rentang label.

In [ ]:
METRICS = ["r2", "spearman", "mae", "rmse", "qwk", "accuracy"]
HIGHER_IS_BETTER = {"r2", "spearman", "qwk", "accuracy"}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_dataset(df, seed):
    X = df.drop(columns="y")
    y = df["y"].astype(int).to_numpy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=seed
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def expected_class(probabilities):
    return probabilities @ np.arange(probabilities.shape[1])


def evaluate_ordinal(y_true, probabilities):
    score = expected_class(probabilities)
    prediction = np.clip(
        np.rint(score), 0, probabilities.shape[1] - 1
    ).astype(int)
    return {
        "r2": r2_score(y_true, score),
        "spearman": spearmanr(y_true, score).statistic,
        "mae": mean_absolute_error(y_true, score),
        "rmse": mean_squared_error(y_true, score) ** 0.5,
        "qwk": cohen_kappa_score(y_true, prediction, weights="quadratic"),
        "accuracy": accuracy_score(y_true, prediction),
    }


def summarize(results, include_std=True):
    columns = METRICS + ["training_seconds", "inference_seconds"]
    grouped = results.groupby(["dataset", "model"])[columns]
    if include_std:
        summary = grouped.agg(["mean", "std"])
    else:
        summary = grouped.mean()
        return summary.reset_index().round(4)
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return summary.reset_index().round(4)


def mean_std_table(results):
    mean = results.groupby(["dataset", "model"])[METRICS].mean()
    std = results.groupby(["dataset", "model"])[METRICS].std()
    table = mean.copy().astype(object)
    for metric in METRICS:
        table[metric] = [
            f"{m:.4f} ± {s:.4f}" if pd.notna(s) else f"{m:.4f}"
            for m, s in zip(mean[metric], std[metric])
        ]
    return table.reset_index()

## 5. Trainer TabM 


In [ ]:
from tabm import TabM


BASELINE_TABM_PARAMS = {
    "n_blocks": 3,
    "d_block": 256,
    "lr": 2e-3,
    "weight_decay": 3e-4,
}


def make_loader(X, y, scaler, seed, shuffle):
    X_tensor = torch.tensor(scaler.transform(X), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=TABM_BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        pin_memory=True,
    )


def predict_tabm(model, scaler, X):
    X_tensor = torch.tensor(scaler.transform(X), dtype=torch.float32)
    loader = DataLoader(X_tensor, batch_size=1024, shuffle=False)
    probabilities = []

    model.eval()
    with torch.inference_mode():
        for batch in loader:
            logits = model(batch.to(DEVICE, non_blocking=True))
            probabilities.append(torch.softmax(logits, dim=-1).mean(1).cpu())

    return torch.cat(probabilities).numpy()


def fit_tabm(X_train, y_train, X_val, y_val, n_classes, params, seed, trial=None):
    set_seed(seed)
    scaler = StandardScaler().fit(X_train)
    train_loader = make_loader(X_train, y_train, scaler, seed, shuffle=True)

    model = TabM.make(
        n_num_features=X_train.shape[1],
        cat_cardinalities=[],
        d_out=n_classes,
        n_blocks=params["n_blocks"],
        d_block=params["d_block"],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=params["weight_decay"],
    )

    best_state = None
    best_mae = np.inf
    patience = TABM_PATIENCE

    for epoch in range(TABM_MAX_EPOCHS):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, n_classes),
                y_batch.repeat_interleave(model.k),
            )
            loss.backward()
            optimizer.step()

        val_probabilities = predict_tabm(model, scaler, X_val)
        val_mae = mean_absolute_error(y_val, expected_class(val_probabilities))
        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"TabM | epoch={epoch + 1:03d} | val_MAE={val_mae:.4f}")

        if trial is not None:
            trial.report(val_mae, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = copy.deepcopy(model.state_dict())
            patience = TABM_PATIENCE
        else:
            patience -= 1
            if patience == 0:
                break

    model.load_state_dict(best_state)
    return model, scaler, best_mae, epoch + 1

## 6. Trainer model lainnya

Semua model memakai validation set untuk early stopping. XGBoost dan LightGBM memakai GPU 0; CatBoost memakai dua GPU bila tersedia. RealMLP memakai GPU 0. LightGBM memakai backend GPU OpenCL dari wheel PyPI agar tidak perlu dikompilasi di Kaggle.

In [ ]:
def make_tree_model(model_name, params, n_classes, seed):
    if model_name == "XGBoost":
        return XGBClassifier(
            **params,
            objective="multi:softprob",
            num_class=n_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            device=f"cuda:{GPU_ID}",
            early_stopping_rounds=50,
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == "LightGBM":
        return LGBMClassifier(
            **params,
            objective="multiclass",
            num_class=n_classes,
            device_type="gpu",
            gpu_device_id=GPU_ID,
            gpu_use_dp=False,
            max_bin=63,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        )
    return CatBoostClassifier(
        **params,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        task_type="GPU",
        devices=CATBOOST_DEVICES,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
    )


def fit_tree_model(model_name, params, X_train, y_train, X_val, y_val, n_classes, seed):
    model = make_tree_model(model_name, params, n_classes, seed)
    if model_name == "LightGBM":
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb_early_stopping(50, verbose=False)],
        )
    elif model_name == "CatBoost":
        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=False,
        )
    else:
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model


def fit_predict(model_name, params, split, n_classes, seed):
    X_train, X_val, X_test, y_train, y_val, _ = split
    set_seed(seed)

    started = time.perf_counter()
    if model_name == "TabM":
        model, scaler, _, epochs = fit_tabm(
            X_train, y_train, X_val, y_val, n_classes, params, seed
        )
        device = str(DEVICE)
    elif model_name == "RealMLP-TD-S":
        model = RealMLPClassifier(device=str(DEVICE))
        model.fit(X_train, y_train, X_val, y_val)
        epochs = 256
        device = str(DEVICE)
    else:
        model = fit_tree_model(
            model_name, params, X_train, y_train, X_val, y_val, n_classes, seed
        )
        epochs = getattr(model, "best_iteration_", None)
        device = f"cuda:{CATBOOST_DEVICES}" if model_name == "CatBoost" else str(DEVICE)
    training_seconds = time.perf_counter() - started

    started = time.perf_counter()
    if model_name == "TabM":
        probabilities = predict_tabm(model, scaler, X_test)
    else:
        probabilities = model.predict_proba(X_test)
    inference_seconds = time.perf_counter() - started

    del model
    torch.cuda.empty_cache()
    return probabilities, training_seconds, inference_seconds, device, epochs

## 7. Baseline tanpa tuning — 1 seed

Baseline disimpan terpisah dan tidak diulang lima kali, sehingga dilaporkan sebagai nilai tunggal tanpa mean ± std.

In [ ]:
def run_suite(name, seeds, params_lookup):
    output_path = OUTPUT_DIR / f"{name}_results_per_seed.csv"
    rows = []

    for dataset_name, df in datasets.items():
        n_classes = df["y"].nunique()
        for seed in seeds:
            split = split_dataset(df, seed)
            y_test = split[-1]

            for model_name in ALL_MODELS:
                params = params_lookup.get((dataset_name, model_name), {})
                probabilities, train_s, infer_s, device, epochs = fit_predict(
                    model_name, params, split, n_classes, seed
                )
                row = {
                    "experiment": name,
                    "dataset": dataset_name,
                    "model": model_name,
                    "seed": seed,
                    "device": device,
                    "epochs": epochs,
                    "training_seconds": train_s,
                    "inference_seconds": infer_s,
                    "params": json.dumps(params, sort_keys=True),
                }
                row.update(evaluate_ordinal(y_test, probabilities))
                rows.append(row)
                pd.DataFrame(rows).to_csv(output_path, index=False)
                print(
                    f"RUN | {name} | {dataset_name} | {model_name} | "
                    f"seed={seed} | MAE={row['mae']:.4f} | "
                    f"train={train_s:.2f}s | infer={infer_s:.2f}s | {device}"
                )

    return pd.DataFrame(rows)


baseline_params = {}
for dataset_name in datasets:
    baseline_params[(dataset_name, "TabM")] = BASELINE_TABM_PARAMS
    for model_name in ["XGBoost", "LightGBM", "CatBoost"]:
        baseline_params[(dataset_name, model_name)] = {}

if RUN_BASELINE:
    baseline_results = run_suite("baseline", [BASELINE_SEED], baseline_params)
    baseline_summary = summarize(baseline_results, include_std=False)
    baseline_summary.to_csv(OUTPUT_DIR / "baseline_summary.csv", index=False)
    print("SAVED | baseline CSV files")
    display(baseline_results)
    display(baseline_summary)

## 8. Search space Optuna

minimize validation MAE.

In [ ]:
def suggest_params(trial, model_name):
    if model_name == "TabM":
        zero_decay = trial.suggest_categorical("weight_decay_zero", [True, False])
        return {
            "n_blocks": trial.suggest_int("n_blocks", 1, 5),
            "d_block": trial.suggest_int("d_block", 64, 1024, step=16),
            "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
            "weight_decay": (
                0.0
                if zero_decay
                else trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True)
            ),
        }
    if model_name == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
    if model_name == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 15, 255),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
    return {
        "iterations": trial.suggest_int("iterations", 200, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 10.0
        ),
    }


def materialize_best_params(model_name, trial_params):
    params = dict(trial_params)
    if model_name == "TabM":
        zero_decay = params.pop("weight_decay_zero")
        if zero_decay:
            params["weight_decay"] = 0.0
    return params


def tuning_objective(trial, model_name, split, n_classes):
    X_train, X_val, _, y_train, y_val, _ = split
    params = suggest_params(trial, model_name)
    started = time.perf_counter()

    if model_name == "TabM":
        _, _, val_mae, _ = fit_tabm(
            X_train,
            y_train,
            X_val,
            y_val,
            n_classes,
            params,
            TUNING_SEED,
            trial,
        )
    else:
        model = fit_tree_model(
            model_name,
            params,
            X_train,
            y_train,
            X_val,
            y_val,
            n_classes,
            TUNING_SEED,
        )
        val_mae = mean_absolute_error(
            y_val, expected_class(model.predict_proba(X_val))
        )
        del model
        torch.cuda.empty_cache()

    trial.set_user_attr("training_seconds", time.perf_counter() - started)
    return val_mae

## 9. Jalankan tuning

Optuna berjalan di memori dan menyimpan ringkasan seluruh trial ke CSV. Jika kernel di-restart, tuning dimulai ulang.

In [ ]:
def run_tuning():
    best = {}

    for dataset_name, df in datasets.items():
        split = split_dataset(df, TUNING_SEED)
        n_classes = df["y"].nunique()

        for model_name in TUNABLE_MODELS:
            study = optuna.create_study(
                direction="minimize",
                sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
                pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
            )

            def show_trial(study, trial):
                if trial.value is not None:
                    print(
                        f"TUNE | {dataset_name} | {model_name} | "
                        f"trial={trial.number + 1}/{N_TRIALS} | "
                        f"val_MAE={trial.value:.4f}"
                    )

            study.optimize(
                lambda trial: tuning_objective(
                    trial, model_name, split, n_classes
                ),
                n_trials=N_TRIALS,
                callbacks=[show_trial],
            )

            params = materialize_best_params(
                model_name, study.best_params
            )
            best[(dataset_name, model_name)] = params
            study.trials_dataframe().to_csv(
                OUTPUT_DIR / f"optuna_{dataset_name}_{model_name}.csv",
                index=False,
            )
            print(
                f"{dataset_name:22} | {model_name:10} | "
                f"best validation MAE={study.best_value:.4f}"
            )

    serializable = {
        f"{dataset}::{model}": params
        for (dataset, model), params in best.items()
    }
    (OUTPUT_DIR / "best_hyperparameters.json").write_text(
        json.dumps(serializable, indent=2)
    )
    return best


def load_best_params():
    data = json.loads(
        (OUTPUT_DIR / "best_hyperparameters.json").read_text()
    )
    return {
        tuple(key.split("::", 1)): params
        for key, params in data.items()
    }


best_params = run_tuning() if RUN_TUNING else load_best_params()

## 10. Evaluasi final best hyperparameter — 5 seed

Empat model tuned memakai best parameter yang sama pada lima split seed. RealMLP tidak memiliki search space, sehingga tetap memakai konfigurasi resminya sebagai pembanding.

In [ ]:
final_params = dict(best_params)
for dataset_name in datasets:
    final_params[(dataset_name, "RealMLP-TD-S")] = {}

if RUN_FINAL:
    tuned_results = run_suite("tuned", FINAL_SEEDS, final_params)
    tuned_summary = summarize(tuned_results)
    tuned_table = mean_std_table(tuned_results)

    tuned_summary.to_csv(OUTPUT_DIR / "tuned_summary.csv", index=False)
    tuned_table.to_csv(OUTPUT_DIR / "tuned_mean_std.csv", index=False)
    print("SAVED | tuned CSV files")

    display(tuned_results)
    display(tuned_table)

## 11. Visualisasi metrik

Setiap metrik memakai heatmap terpisah agar skala tidak tercampur. Warna hijau lebih pekat berarti lebih baik; anotasi final menampilkan mean ± std.

In [ ]:
METRIC_LABELS = {
    "r2": "R² ↑",
    "spearman": "Spearman ↑",
    "mae": "MAE ↓",
    "rmse": "RMSE ↓",
    "qwk": "QWK ↑",
    "accuracy": "Accuracy ↑",
}


def plot_metrics(results, title, filename):
    grouped = results.groupby(["dataset", "model"])[METRICS]
    means = grouped.mean()
    stds = grouped.std()
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    for metric, ax in zip(METRICS, axes.flat):
        matrix = means[metric].unstack("model")
        deviation = stds[metric].unstack("model")
        annotation = matrix.copy().astype(object)
        for row in matrix.index:
            for column in matrix.columns:
                mean = matrix.loc[row, column]
                std = deviation.loc[row, column]
                annotation.loc[row, column] = (
                    f"{mean:.3f}\n±{std:.3f}"
                    if pd.notna(std)
                    else f"{mean:.3f}"
                )
        cmap = "YlGn" if metric in HIGHER_IS_BETTER else "YlGn_r"
        sns.heatmap(
            matrix,
            annot=annotation,
            fmt="",
            cmap=cmap,
            linewidths=0.5,
            cbar=True,
            ax=ax,
        )
        ax.set_title(METRIC_LABELS[metric])
        ax.set_xlabel("")
        ax.set_ylabel("")

    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(OUTPUT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()


if RUN_BASELINE:
    plot_metrics(baseline_results, "Baseline — 1 seed", "baseline_metrics.png")

if RUN_FINAL:
    plot_metrics(
        tuned_results,
        "Best hyperparameter — mean 5 seed",
        "tuned_metrics.png",
    )

## Output

Semua file berada di folder results/:

- baseline_results_per_seed.csv: baseline 1 seed lengkap.
- baseline_summary.csv: ringkasan baseline satu seed tanpa std.
- optuna_<dataset>_<model>.csv: seluruh trial Optuna.
- best_hyperparameters.json: parameter terbaik.
- tuned_results_per_seed.csv: hasil final 5 seed lengkap.
- tuned_summary.csv: mean dan std numerik untuk lima seed.
- tuned_mean_std.csv: tabel siap laporan.
- baseline_metrics.png dan tuned_metrics.png: visualisasi enam metrik.